# 04. 미세스크래치 분류로직 검증

현재는 실제 micro/scratch label이 없으므로 precision/recall을 계산하지 않는다. 대신 metric별 quantile threshold 후보를 보고, 실제 데이터 calibration 시 어떤 지표를 쓸지 비교한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

In [ ]:
feature_path = RUNS_ROOT / "component_features.csv"
if not feature_path.exists():
    manifest = generate_random_scratch_dataset(n_samples=120, size=640, seed=7, overwrite=True)
    features = extract_feature_table(manifest, min_area=12)
    features.to_csv(feature_path, index=False, encoding="utf-8-sig")
else:
    features = pd.read_csv(feature_path)

metric_cols = [
    "luma_contrast_abs",
    "luma_contrast_z",
    "rgb_euclidean_contrast",
    "rgb_contrast_z",
    "max_channel_contrast_abs",
    "mean_channel_contrast_abs",
]
candidate_tables = []
for metric in metric_cols:
    candidate_tables.append(candidate_thresholds_by_quantile(features, metric=metric))
candidates = pd.concat(candidate_tables, ignore_index=True)
candidates.to_csv(RUNS_ROOT / "candidate_thresholds_by_metric.csv", index=False, encoding="utf-8-sig")
display(candidates)

## 임시 후보 분류 예시

In [ ]:
metric = "rgb_euclidean_contrast"
threshold = float(features[metric].quantile(0.30))
pred = classify_components(features, metric=metric, contrast_threshold=threshold, width_threshold=None)
pred.to_csv(RUNS_ROOT / "component_candidate_predictions.csv", index=False, encoding="utf-8-sig")
print("metric:", metric)
print("temporary 30% quantile threshold:", threshold)
display(pred["pred_label"].value_counts().reset_index(name="count"))
display(classification_report(pred))

## 낮은 contrast 후보 예시

In [ ]:
low = pred.sort_values(metric).head(8)
manifest = pd.read_csv(DATA_ROOT / "metadata" / "samples.csv").set_index("sample_id")
fig, axes = plt.subplots(len(low), 2, figsize=(7, 2.2 * len(low)))
for row_idx, (_, row) in enumerate(low.iterrows()):
    meta = manifest.loc[row["sample_id"]]
    img = load_image(meta["image_path"])
    mask = load_mask(meta["mask_path"])
    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title(f'{row["sample_id"]} | rgb={row[metric]:.1f}, luma={row["luma_contrast_abs"]:.1f}')
    axes[row_idx, 0].axis("off")
    axes[row_idx, 1].imshow(mask, cmap="gray")
    axes[row_idx, 1].set_title(f'w={row["width_px"]:.1f}px, angle={row["orientation_deg"]:.1f}')
    axes[row_idx, 1].axis("off")
plt.tight_layout()
plt.savefig(RUNS_ROOT / "low_contrast_candidate_examples.png", dpi=150)
plt.show()